# 04. ConversationEntityMemory / ConversationKGMemory → LangGraph **Store** (장기 기억)

| legacy | LangGraph |
|---|---|
| `ConversationEntityMemory(llm=llm)` — 개체(사람/회사) 별 요약 저장 | `Store` 의 `("entities", user_id)` 네임스페이스에 `key=개체명, value={"summary": ...}` |
| `ConversationKGMemory(llm=llm)` — (주어, 관계, 목적어) 트리플 저장 | `Store` 의 `("kg", user_id)` 네임스페이스에 트리플 저장 |
| `conversation.memory.entity_store.store` 로 조회 | `store.search(namespace)` 로 조회 |
| 대화(메모리 객체)마다 따로 존재 | **thread 가 달라도 같은 user 면 공유** (장기 기억) |

> Entity/KG 메모리는 1:1 대체 클래스가 **없습니다**. 대신 "무엇을 추출해 어디에 저장할지"를 직접 정의하고, 저장소로 `Store` 를 씁니다.
> 추출은 `llm.with_structured_output(Pydantic 모델)` 로 구현합니다.
>
> * **Checkpointer (단기 기억)**: thread 하나의 대화 기록
> * **Store (장기 기억)**: thread 를 넘어 유지되는 사실/프로필/지식

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. 추출 스키마 정의 (Entity + KG 트리플)

In [2]:
from pydantic import BaseModel, Field


class Entity(BaseModel):
    name: str = Field(description="사람/회사/장소 등 개체의 이름 (예: 테디)")
    summary: str = Field(description="이 개체에 대해 지금까지 알려진 사실 요약. 기존 요약이 있으면 새 정보와 합쳐서 작성")


class Triple(BaseModel):
    subject: str = Field(description="주어 개체 이름")
    predicate: str = Field(description="관계 (예: 직업, 동료, 계획)")
    object: str = Field(description="목적어")


class MemoryExtraction(BaseModel):
    entities: list[Entity]
    triples: list[Triple]


extractor = llm.with_structured_output(MemoryExtraction)

## 2. 그래프: `write_memory` (추출·저장) → `call_model` (관련 기억을 읽어 답변)
노드에서는 `runtime.store` 로 Store 에, `runtime.context` 로 실행 시 넘긴 `user_id` 에 접근합니다.

In [3]:
from dataclasses import dataclass
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.runtime import Runtime


@dataclass
class Context:
    user_id: str


EXTRACT_PROMPT = """사용자 메시지에서 개체(entity)와 관계(triple)를 추출하세요.
이미 알고 있는 개체 요약은 다음과 같습니다. 같은 개체가 다시 나오면 기존 요약에 새 정보를 합치세요.
{existing}"""


def write_memory(state: MessagesState, runtime: Runtime[Context]):
    user_id = runtime.context.user_id
    ns_entity, ns_kg = ("entities", user_id), ("kg", user_id)

    existing = {it.key: it.value["summary"] for it in runtime.store.search(ns_entity, limit=100)}
    last_user = state["messages"][-1].content
    result = extractor.invoke([SystemMessage(EXTRACT_PROMPT.format(existing=existing or "(없음)")),
                               HumanMessage(last_user)])

    for e in result.entities:          # legacy: entity_store 갱신
        runtime.store.put(ns_entity, e.name, {"summary": e.summary})
    for t in result.triples:           # legacy: KG 에 트리플 추가
        runtime.store.put(ns_kg, f"{t.subject}|{t.predicate}|{t.object}", t.model_dump())
    return {}


def call_model(state: MessagesState, runtime: Runtime[Context]):
    user_id = runtime.context.user_id
    question = state["messages"][-1].content

    # legacy 처럼 "현재 입력에 등장한 개체"에 대한 기억만 골라서 프롬프트에 넣는다
    entities = [it for it in runtime.store.search(("entities", user_id), limit=100) if it.key in question]
    triples = [it.value for it in runtime.store.search(("kg", user_id), limit=100)
               if it.value["subject"] in question or it.value["object"] in question]

    context = "\n".join(f"- {e.key}: {e.value['summary']}" for e in entities) or "(없음)"
    context += "\n" + "\n".join(f"- ({t['subject']}, {t['predicate']}, {t['object']})" for t in triples)
    system = SystemMessage(f"당신은 친절한 비서입니다. 아래 기억(Context)만 사실로 사용하고, 없으면 모른다고 답하세요.\n\nContext:\n{context}")
    return {"messages": [llm.invoke([system] + state["messages"])]}


store = InMemoryStore()
graph = (
    StateGraph(MessagesState, context_schema=Context)
    .add_node("write_memory", write_memory)
    .add_node("call_model", call_model)
    .add_edge(START, "write_memory")
    .add_edge("write_memory", "call_model")
    .add_edge("call_model", END)
    .compile(checkpointer=InMemorySaver(), store=store)
)

## 3. 대화하기 (legacy `conversation.predict(input=...)`)

In [4]:
out = graph.invoke(
    {"messages": [HumanMessage(
        "테디와 셜리는 한 회사에서 일하는 동료입니다. 테디는 개발자이고 셜리는 디자이너입니다. "
        "그들은 최근 회사에서 일하는 것을 그만두고 자신들의 회사를 차릴 계획을 세우고 있습니다."
    )]},
    config={"configurable": {"thread_id": "conv-1"}},
    context={"user_id": "user-1"},
)
print(out["messages"][-1].content)

맞습니다. 테디는 개발자이고 셜리는 디자이너로, 둘은 함께 한 회사에서 일하는 동료입니다. 그들은 최근에 회사를 그만두고 자신들의 회사를 차릴 계획을 세우고 있습니다.


## 4. 저장된 기억 확인 (legacy `conversation.memory.entity_store.store`)

In [5]:
print("[Entity]")
for it in store.search(("entities", "user-1")):
    print(f"  {it.key}: {it.value['summary']}")

print("[Knowledge Graph]")
for it in store.search(("kg", "user-1")):
    print(f"  ({it.value['subject']}) -[{it.value['predicate']}]-> ({it.value['object']})")

[Entity]
  테디: 개발자이며, 셜리와 함께 한 회사에서 일하는 동료이다.
  셜리: 디자이너이며, 테디와 함께 한 회사에서 일하는 동료이다.
  회사: 테디와 셜리가 함께 일하던 곳이다.
  자신들의 회사: 테디와 셜리가 차릴 계획을 세우고 있는 회사이다.
[Knowledge Graph]
  (테디) -[직업]-> (개발자)
  (셜리) -[직업]-> (디자이너)
  (테디와 셜리) -[같이 일하는]-> (회사)
  (테디와 셜리) -[차릴 계획]-> (자신들의 회사)


## 5. 장기 기억: **새 대화(thread)** 에서도 기억한다
legacy Entity/KG 메모리는 메모리 객체가 사라지면 함께 사라졌습니다. Store 는 thread 와 독립적입니다.

In [6]:
out = graph.invoke(
    {"messages": [HumanMessage("셜리의 직업은 뭐고, 테디와는 어떤 사이야?")]},
    config={"configurable": {"thread_id": "conv-2"}},   # 새 대화
    context={"user_id": "user-1"},                       # 같은 사용자
)
print("[user-1 / 새 thread]", out["messages"][-1].content)

out = graph.invoke(
    {"messages": [HumanMessage("셜리의 직업은 뭐고, 테디와는 어떤 사이야?")]},
    config={"configurable": {"thread_id": "conv-3"}},
    context={"user_id": "user-2"},                       # 다른 사용자 → 네임스페이스가 달라 모름
)
print("[user-2]", out["messages"][-1].content)

[user-1 / 새 thread] 셜리의 직업은 디자이너이며, 테디와는 동료입니다.


[user-2] 죄송하지만, 셜리의 직업이나 테디와의 관계에 대한 구체적인 정보는 가지고 있지 않습니다.


### 참고
* `InMemoryStore` 는 프로세스가 끝나면 사라집니다. 운영에서는 `PostgresStore` (`langgraph-checkpoint-postgres`) 등 영속 Store 를 사용합니다.
* 개체 매칭을 문자열 포함(`it.key in question`)으로 단순화했습니다. 더 정교하게 하려면 05번 노트북처럼 Store 의 **시맨틱 검색**을 사용하면 됩니다.
* 이런 추출/갱신 로직을 라이브러리로 제공하는 것이 `langmem` 패키지(`create_memory_manager` 등)입니다.